# ⚽ FPL AI Squad Planner: Step-by-Step

### สิ่งที่จะได้เห็นในคลิปนี้:
1. **Environment Setup**: ตรวจสอบการเชื่อมต่อกับ Ollama (`deepseek-v4.1-flash:cloud`)
2. **Vision Extraction (Step 1)**: ส่งภาพ Screenshot หน้าจัดทีม FPL เข้า Vision Model แล้วแปลงเป็น Structured JSON (Pydantic)
3. **FPL API Query (Step 2)**: ทดสอบ Custom Tool ดึงข้อมูลสถิติจริงจาก Official Fantasy Premier League API
4. **CrewAI Multi-Agent Execution (Step 3)**: รันทีม AI วิเคราะห์อาการบาดเจ็บ วางแผนซื้อขาย และจัด 11 ตัวจริงพร้อมกัปตัน
5. **Result Presentation**: สรุปผลลัพธ์พร้อมนำไปจัดทีมจริง

## 🛠️ Step 0: ตรวจสอบ Ollama Models

In [1]:
import ollama

# ตรวจสอบว่าโมเดลพร้อมใช้งานบน Ollama หรือไม่
try:
    models_list = ollama.list()
    model_names = [m.get('model', m.get('name', '')) for m in models_list.get('models', [])]
    print("📦 Available Models in Ollama:")
    for name in model_names:
        print(f" - {name}")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการเชื่อมต่อ Ollama: {e}")
    print("อย่าลืมรันคำสั่ง: ollama serve และ pull โมเดลที่ต้องใช้")

📦 Available Models in Ollama:
 - gemma4:31b-cloud
 - glm-5.3:cloud
 - deepseek-v4.1-flash:cloud
 - gpt-oss:120b-cloud
 - embeddinggemma-2:570m
 - gpt-oss:20b-cloud
 - llava:7b
 - deepseek-r1:8b
 - qwen3.5:9b
 - gemma4:e4b
 - llama3.2:3b


## 🧩 Step 1: กำหนด Pydantic Schema สำหรับข้อมูลทีม FPL
กำหนดโครงสร้างข้อมูลที่ต้องการให้ Vision Model แกะออกมาจากรูปภาพ Screenshot

In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional

class PlayerInfo(BaseModel):
    name: str = Field(description="ชื่อนักเตะ เช่น Haaland, Calafiori, João Pedro")
    position: str = Field(description="ตำแหน่ง: GKP, DEF, MID, FWD")
    price: float = Field(description="ราคาขายที่แสดงบนการ์ด เช่น 15.6, 7.6, 5.7")
    fixture: str = Field(description="คู่แข่งและสถานะ เช่น BHA (H), MCI (A)")
    status_flag: Optional[str] = Field(default="NONE", description="สถานะแจ้งเตือน: RED (เจ็บ), YELLOW (เช็คฟิต), หรือ NONE")

class FPLTeamState(BaseModel):
    gameweek: str = Field(description="Gameweek ปัจจุบัน เช่น Gameweek 9")
    deadline: str = Field(description="เส้นตายการจัดทีม เช่น Sat 31 Oct, 18:00")
    budget_itb: float = Field(description="เงินคงเหลือในธนาคาร (In The Bank) เป็น £m เช่น 1.3")
    free_transfers: int = Field(description="จำนวน Free Transfers ที่มี เช่น 1")
    cost_pts: int = Field(default=0, description="แต้มหักจากการย้ายตัวปัจจุบัน")
    wildcard_available: bool = Field(description="ปุ่ม Wildcard ยังใช้ได้หรือไม่")
    free_hit_available: bool = Field(description="ปุ่ม Free Hit ยังใช้ได้หรือไม่")
    goalkeepers: List[PlayerInfo] = Field(description="ผู้รักษาประตู 2 คน")
    defenders: List[PlayerInfo] = Field(description="กองหลัง 5 คน")
    midfielders: List[PlayerInfo] = Field(description="กองกลาง 5 คน")
    forwards: List[PlayerInfo] = Field(description="กองหน้า 3 คน")

print("✅ Pydantic Schemas defined successfully!")

✅ Pydantic Schemas defined successfully!


## 📸 Step 2: ทดสอบ Image-to-JSON Extractor ด้วย Vision Model
นำภาพ Screenshot ใส่เข้าไป แล้วให้ Vision Model แกะข้อมูลตามโครงสร้าง `FPLTeamState`

In [3]:
from pathlib import Path

# -----------------------------------------------------
# กำหนด Path รูปภาพ Screenshot ของคุณ
# -----------------------------------------------------
IMAGE_PATH = "./fpl_screenshot.png"  # เปลี่ยนเป็น path รูปของคุณ

image_file = Path(IMAGE_PATH).expanduser()
if image_file.is_file():
    print(f"🖼️  พบไฟล์รูป: {image_file.resolve()}")
else:
    print(f"⚠️  ยังไม่พบไฟล์รูปที่ {image_file.resolve()}")
    print("    ให้วางไฟล์ fpl_screenshot.png ไว้ในโฟลเดอร์เดียวกับ notebook นี้")
    print("    (หรือแก้ค่า IMAGE_PATH ให้ชี้ไปยังรูปจริง)")

IMAGE_PATH


🖼️  พบไฟล์รูป: F:\Udemy\youtube\agent-ai\crew-ai\3_fantasy\fpl_screenshot.png


'./fpl_screenshot.png'

In [4]:
import base64
import json
import re
import time
from pathlib import Path
from typing import List, Optional

from ollama import Client, ResponseError
from pydantic import ValidationError

ROWS = {"goalkeepers": "GKP", "defenders": "DEF", "midfielders": "MID", "forwards": "FWD"}
POS_ALIASES = {
    "GK": "GKP", "GKP": "GKP", "GOALKEEPER": "GKP",
    "DEF": "DEF", "DEFENDER": "DEF",
    "MID": "MID", "MIDFIELDER": "MID",
    "FWD": "FWD", "FORWARD": "FWD", "ST": "FWD", "ATT": "FWD",
}
FLAG_KEYS = ("status_flag", "flag", "status", "warning", "injury_status", "alert")


def resolve_image_path(path):
    """หาไฟล์รูปจาก path ที่ระบุ หรือจากโฟลเดอร์ fantasy/ ที่อยู่ข้าง notebook"""
    candidate = Path(path).expanduser()
    if candidate.is_file():
        return candidate
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd() / "fantasy"):
        resolved = base / candidate
        if resolved.is_file():
            return resolved
    raise FileNotFoundError(f"ไม่พบไฟล์รูปภาพ: {candidate.resolve()}")


def strip_code_fences(text):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def first_json_object(text):
    """ดึง JSON object ก้อนแรกออกจากข้อความ ตัด prose หรือโค้ดบล็อกที่โมเดลอาจห่อมาให้"""
    text = strip_code_fences(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    if start == -1:
        raise ValueError("model output contains no JSON object")

    depth, in_string, escaped = 0, False, False
    for i, char in enumerate(text[start:], start):
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
        elif char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise ValueError("model output has an unterminated JSON object")


def to_number(value, default=0.0):
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        match = re.search(r"-?\d+(?:\.\d+)?", value.replace(",", ""))
        if match:
            return float(match.group())
    return default


def to_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "yes", "y", "available", "play", "1"}


def to_flag(value):
    text = str(value or "").upper()
    if "RED" in text or "INJUR" in text:
        return "RED"
    if "YELLOW" in text or "DOUBT" in text or "WARN" in text or "FIT" in text:
        return "YELLOW"
    return "NONE"


def to_gameweek(value):
    match = re.search(r"\d+", str(value or ""))
    return f"Gameweek {match.group()}" if match else str(value or "").strip()


def flatten_state(raw):
    """รองรับกรณีโมเดลห่อข้อมูลเป็น {header: ..., players: {...}} แทน schema แบบแบน"""
    if not isinstance(raw, dict):
        raise ValueError("model output is not a JSON object")
    if "header" not in raw and "players" not in raw:
        return raw

    flat = dict(raw.get("header") or {})
    players = raw.get("players") or {}
    for key, code in ROWS.items():
        if key in players:
            flat.setdefault(key, players[key])
        if code in players:
            flat.setdefault(key, players[code])

    for key in ("wildcard_available", "free_hit_available"):
        chip = flat.get(key)
        if isinstance(chip, dict):
            flat[key] = chip.get("status") or chip.get("available")
        if flat.get(key) is None:
            alias = key.replace("_available", "")
            if alias in flat:
                chip = flat[alias]
                flat[key] = (chip.get("status") or chip.get("available")) if isinstance(chip, dict) else chip

    for alias in ("bank", "budget", "itb", "budget_in_the_bank"):
        if "budget_itb" not in flat and alias in flat:
            flat["budget_itb"] = flat[alias]
    if "gw" in flat and "gameweek" not in flat:
        flat["gameweek"] = flat["gw"]

    return flat


def normalize_state(raw):
    """ปรับชนิดข้อมูลและชื่อคีย์ให้ตรงกับ FPLTeamState เช่น "£1.3m" -> 1.3"""
    flat = flatten_state(raw)
    state = {
        "gameweek": to_gameweek(flat.get("gameweek")),
        "deadline": str(flat.get("deadline") or "").strip(),
        "budget_itb": round(to_number(flat.get("budget_itb")), 1),
        "free_transfers": int(to_number(flat.get("free_transfers"))),
        "cost_pts": int(to_number(flat.get("cost_pts"))),
        "wildcard_available": to_bool(flat.get("wildcard_available")),
        "free_hit_available": to_bool(flat.get("free_hit_available")),
    }

    for key, default_position in ROWS.items():
        players = flat.get(key) or []
        if isinstance(players, dict):
            players = list(players.values())
        state[key] = [
            {
                "name": str(player.get("name") or "").strip(),
                "position": POS_ALIASES.get(str(player.get("position") or "").strip().upper(), default_position),
                "price": round(to_number(player.get("price")), 1),
                "fixture": str(player.get("fixture") or "").strip(),
                "status_flag": to_flag(next((player[k] for k in FLAG_KEYS if player.get(k) is not None), None)),
            }
            for player in players
            if isinstance(player, dict)
        ]

    return state


def parse_team_state(text):
    return FPLTeamState.model_validate(normalize_state(first_json_object(text)))


FPL_SCHEMA = json.dumps(FPLTeamState.model_json_schema(), ensure_ascii=False)

EXTRACT_PROMPT = f"""Extract the Fantasy Premier League (FPL) squad from this screenshot.

Return ONLY a raw JSON object matching this JSON Schema. Use the schema's top-level keys
directly - never nest the values under "header" or "players":

{FPL_SCHEMA}

Rules:
- gameweek: string, e.g. "Gameweek 9".
- deadline: string exactly as printed, e.g. "Sat 31 Oct, 18:00".
- budget_itb: float in millions, e.g. 1.3 (strip the pound sign and the "m").
- free_transfers and cost_pts: integers.
- wildcard_available / free_hit_available: true when the chip still shows "Play".
- goalkeepers (2), defenders (5), midfielders (5), forwards (3): pitch rows top to bottom.
- position: exactly one of GKP, DEF, MID, FWD.
- price: float in millions.
- fixture: the fixture text on the card, e.g. "BHA (H)".
- status_flag: "RED" for a red exclamation mark, "YELLOW" for a yellow warning
  triangle, otherwise "NONE".

Output raw JSON only. No prose, no markdown, no code fences."""

REPAIR_SYSTEM = (
    "You convert scouting notes into strict JSON. Output raw JSON only - no prose, "
    "no markdown. Use the JSON Schema's top-level keys directly, do not nest them."
)


def test_vision_extraction(image_path, model_name="deepseek-v4.1-flash:cloud", host="http://localhost:11434"):
    """สกัดข้อมูลจากภาพเป็น FPLTeamState พร้อมซ่อมผลลัพธ์ที่ไม่ใช่ JSON ให้อัตโนมัติ"""
    image_file = resolve_image_path(image_path)
    client = Client(host=host)
    schema = FPLTeamState.model_json_schema()
    image_b64 = base64.b64encode(image_file.read_bytes()).decode()

    started = time.time()
    reply = client.chat(
        model=model_name,
        messages=[{"role": "user", "content": EXTRACT_PROMPT, "images": [image_b64]}],
        format=schema,
        options={"temperature": 0.0},
    )
    raw_text = reply.message.content or ""
    print(f"   pass 1 (vision): {time.time() - started:.1f}s, {len(raw_text)} chars")

    try:
        return parse_team_state(raw_text)
    except (ValueError, ValidationError) as exc:
        reason = str(exc).splitlines()[0][:120]
        print(f"   pass 1 ได้คำตอบที่ไม่ใช่ JSON ({reason}) -> ส่งซ่อมเป็น JSON")

    # Pass 2: ให้โมเดลแปลงข้อความที่ได้ให้เป็น JSON ตาม schema (แชทแบบข้อความล้วน)
    started = time.time()
    reply = client.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": REPAIR_SYSTEM},
            {
                "role": "user",
                "content": (
                    "Convert these FPL notes into JSON matching this JSON Schema exactly "
                    f"(use its top-level keys, do not nest them):\n{FPL_SCHEMA}\n\nNOTES:\n{raw_text}"
                ),
            },
        ],
        format=schema,
        options={"temperature": 0.0},
    )
    print(f"   pass 2 (repair): {time.time() - started:.1f}s")
    return parse_team_state(reply.message.content or "")


try:
    extracted_team = test_vision_extraction(IMAGE_PATH)
    print("🎉 สกัดข้อมูลสำเร็จจากรูปจริง!")
    print(json.dumps(extracted_team.model_dump(), indent=2, ensure_ascii=False))
except FileNotFoundError as e:
    print(f"📁 {e}")
except ResponseError as e:
    print(f"🔌 เชื่อมต่อ Ollama ไม่สำเร็จ: {e}")
except (ValueError, ValidationError) as e:
    print(f"🧩 โมเดลตอบกลับมาแต่ไม่ใช่ JSON ที่ใช้ได้: {str(e).splitlines()[0]}")
except Exception as e:
    print(f"ℹ️ สกัดข้อมูลจากรูปไม่สำเร็จ: {type(e).__name__}: {e}")

if "extracted_team" not in globals():
    print("➡️ สลับไปใช้ Mock Data (ตรงกับภาพ Gameweek 9 ตัวอย่าง) เพื่อสาธิตต่อในขั้นตอนถัดไป")

    mock_data = {
        "gameweek": "Gameweek 9",
        "deadline": "Sat 31 Oct, 18:00",
        "budget_itb": 1.3,
        "free_transfers": 1,
        "cost_pts": 0,
        "wildcard_available": True,
        "free_hit_available": True,
        "goalkeepers": [
            {"name": "Verbruggen", "position": "GKP", "price": 4.5, "fixture": "MCI (A)", "status_flag": "NONE"},
            {"name": "Kinsky", "position": "GKP", "price": 4.4, "fixture": "CRY (H)", "status_flag": "NONE"},
        ],
        "defenders": [
            {"name": "Calafiori", "position": "DEF", "price": 5.7, "fixture": "LIV (A)", "status_flag": "YELLOW"},
            {"name": "Ajer", "position": "DEF", "price": 4.5, "fixture": "NFO (H)", "status_flag": "NONE"},
            {"name": "O'Shea", "position": "DEF", "price": 4.0, "fixture": "HUL (A)", "status_flag": "NONE"},
            {"name": "Hall", "position": "DEF", "price": 5.3, "fixture": "EVE (H)", "status_flag": "NONE"},
            {"name": "Aina", "position": "DEF", "price": 4.3, "fixture": "BRE (A)", "status_flag": "NONE"},
        ],
        "midfielders": [
            {"name": "Schade", "position": "MID", "price": 6.2, "fixture": "NFO (H)", "status_flag": "NONE"},
            {"name": "B.Fernandes", "position": "MID", "price": 11.7, "fixture": "CHE (A)", "status_flag": "NONE"},
            {"name": "Mbeumo", "position": "MID", "price": 7.9, "fixture": "CHE (A)", "status_flag": "NONE"},
            {"name": "I.Sangaré", "position": "MID", "price": 4.7, "fixture": "BRE (A)", "status_flag": "NONE"},
            {"name": "M.Sangaré", "position": "MID", "price": 5.4, "fixture": "NFO (H)", "status_flag": "NONE"},
        ],
        "forwards": [
            {"name": "Kostoulas", "position": "FWD", "price": 5.7, "fixture": "MCI (A)", "status_flag": "NONE"},
            {"name": "João Pedro", "position": "FWD", "price": 7.6, "fixture": "MUN (H)", "status_flag": "RED"},
            {"name": "Haaland", "position": "FWD", "price": 15.6, "fixture": "BHA (H)", "status_flag": "NONE"},
        ],
    }
    extracted_team = FPLTeamState.model_validate(mock_data)
    print("✅ โหลด Mock Data สำเร็จ:")
    print(f"- Gameweek: {extracted_team.gameweek}")
    print(f"- ITB: £{extracted_team.budget_itb}m, Free Transfers: {extracted_team.free_transfers}")
    print(f"- มีตัวเจ็บ: {[p.name for p in extracted_team.forwards if p.status_flag == 'RED']}")


   pass 1 (vision): 9.8s, 1559 chars
🎉 สกัดข้อมูลสำเร็จจากรูปจริง!
{
  "gameweek": "Gameweek 9",
  "deadline": "Sat 31 Oct, 18:00",
  "budget_itb": 1.3,
  "free_transfers": 1,
  "cost_pts": 0,
  "wildcard_available": true,
  "free_hit_available": true,
  "goalkeepers": [
    {
      "name": "Verbruggen",
      "position": "GKP",
      "price": 4.5,
      "fixture": "MCI (A)",
      "status_flag": "NONE"
    },
    {
      "name": "Kinsky",
      "position": "GKP",
      "price": 4.4,
      "fixture": "CRY (H)",
      "status_flag": "NONE"
    }
  ],
  "defenders": [
    {
      "name": "Calafiori",
      "position": "DEF",
      "price": 5.7,
      "fixture": "LIV (A)",
      "status_flag": "YELLOW"
    },
    {
      "name": "Ajer",
      "position": "DEF",
      "price": 4.5,
      "fixture": "NFO (H)",
      "status_flag": "NONE"
    },
    {
      "name": "O'Shea",
      "position": "DEF",
      "price": 4.0,
      "fixture": "HUL (A)",
      "status_flag": "NONE"
    },
    {
    

## 🌐 Step 3: ทดสอบเชื่อมต่อ Official FPL API Tool
ทดสอบว่า Custom Tool สามารถดึงรายชื่อนักเตะ ฟอร์มการเล่น และค่า Expected Points (ep_next) จาก API จริงได้ถูกต้องหรือไม่

In [5]:
import requests
from crewai.tools import tool

@tool("Query FPL Replacements")
def fetch_fpl_market_data(position: str, max_price: float) -> str:
    """ดึงข้อมูลนักเตะจาก FPL API ตามตำแหน่ง (GKP, DEF, MID, FWD) ที่ราคาไม่เกิน max_price และพร้อมลงเล่น"""
    url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    try:
        res = requests.get(url, timeout=10)
        data = res.json()
        
        pos_map = {"GKP": 1, "DEF": 2, "MID": 3, "FWD": 4}
        target_pos = pos_map.get(position.upper(), 4)
        teams = {t["id"]: t["short_name"] for t in data["teams"]}
        
        candidates = []
        for p in data["elements"]:
            price = p["now_cost"] / 10.0
            if p["element_type"] == target_pos and price <= max_price and p["status"] == "a":
                candidates.append({
                    "name": p["web_name"],
                    "team": teams.get(p["team"], "UNK"),
                    "price": price,
                    "form": float(p["form"]),
                    "ep_next": float(p["ep_next"] or 0),
                    "selected_by": f"{p['selected_by_percent']}%"
                })
        
        top = sorted(candidates, key=lambda x: x["ep_next"], reverse=True)[:8]
        return str(top)
    except Exception as e:
        return f"Error fetching FPL data: {str(e)}"

# ทดสอบรัน Tool หาตัวแทนกองหน้า (FWD) งบไม่เกิน 8.9m (£7.6m + £1.3m)
print("🔍 ทดสอบค้นหากองหน้า FWD งบไม่เกิน 8.9m:")
sample_replacements = fetch_fpl_market_data.run(position="FWD", max_price=8.9)
print(sample_replacements)

🔍 ทดสอบค้นหากองหน้า FWD งบไม่เกิน 8.9m:
[{'name': 'Evanilson', 'team': 'BOU', 'price': 6.4, 'form': 8.5, 'ep_next': 8.5, 'selected_by': '15.6%'}, {'name': 'N.Jackson', 'team': 'AVL', 'price': 7.0, 'form': 8.2, 'ep_next': 8.2, 'selected_by': '23.9%'}, {'name': 'McBurnie', 'team': 'HUL', 'price': 5.5, 'form': 5.8, 'ep_next': 5.8, 'selected_by': '1.7%'}, {'name': 'Thiago', 'team': 'BRE', 'price': 7.8, 'form': 5.5, 'ep_next': 5.5, 'selected_by': '9.2%'}, {'name': 'Kostoulas', 'team': 'BHA', 'price': 5.8, 'form': 4.2, 'ep_next': 4.2, 'selected_by': '10.3%'}, {'name': 'Cherif', 'team': 'COV', 'price': 5.0, 'form': 4.0, 'ep_next': 4.0, 'selected_by': '0.5%'}, {'name': 'Calvert-Lewin', 'team': 'LEE', 'price': 6.1, 'form': 3.8, 'ep_next': 3.8, 'selected_by': '21.2%'}, {'name': 'Marmoush', 'team': 'TOT', 'price': 7.0, 'form': 3.5, 'ep_next': 3.5, 'selected_by': '2.7%'}]


## 🤖 Step 4: ประกอบ CrewAI ด้วยโมเดล `deepseek-v4.1-flash:cloud`
สร้าง Agent 2 ตัว:
1. **Scout Agent**: เช็คอาการบาดเจ็บ และใช้ Tool ดึงนักเตะใหม่ที่คุ้มค่า
2. **Manager Agent**: ตัดสินใจเรื่อง Transfers, Chip, 11 ตัวจริง, และกัปตันทีม

In [6]:
from crewai import Agent, Task, Crew, Process, LLM

# เชื่อมต่อ DeepSeek ผ่าน Ollama
fpl_llm = LLM(
    model="ollama/deepseek-v4.1-flash:cloud",
    base_url="http://localhost:11434",
    temperature=0.2
)

# 1. Scout Agent
scout = Agent(
    role="FPL Data Analyst & Scout",
    goal="ตรวจเช็คสถานะความพร้อมของทีม ค้นหานักเตะที่มีปัญหาบาดเจ็บ และเสนอชื่อตัวแทนที่ดีที่สุดจาก FPL API",
    backstory="ผู้เชี่ยวชาญสถิติ FPL ที่แม่นยำด้าน Expected Points และโปรแกรมการแข่งขัน",
    tools=[fetch_fpl_market_data],
    llm=fpl_llm,
    verbose=True
)

# 2. Manager Agent
manager = Agent(
    role="Elite FPL Manager",
    goal="ตัดสินใจเลือกย้ายตัว จัด 11 ตัวจริง ลำดับตัวสำรอง และเลือกกัปตันทีม",
    backstory="กุนซือ FPL อันดับท็อปที่มีแนวทางรัดกุม ไม่เสียแต้มฟรี และเลือกตัวทำแต้มสูงสุด",
    llm=fpl_llm,
    verbose=True
)

print("✅ CrewAI Agents configured with deepseek-v4.1-flash:cloud!")

✅ CrewAI Agents configured with deepseek-v4.1-flash:cloud!


## 🚀 Step 5: สร้าง Tasks และรัน CrewAI Execution
เซลล์นี้คือจุดไฮไลต์สำหรับอัดหน้าจอคลิป YouTube เพื่อให้ผู้ชมเห็น Output การสนทนาและวิเคราะห์ของ Agent แต่ละตัวแบบสดๆ

In [7]:
import time
from concurrent.futures import ThreadPoolExecutor

team_json_str = json.dumps(extracted_team.model_dump(), indent=2, ensure_ascii=False)

task1 = Task(
    description=f"""
    วิเคราะห์สภาพทีมจากข้อมูล JSON นี้:
    {team_json_str}

    1. ตรวจสอบนักเตะที่ติดสถานะ RED (เจ็บ) หรือ YELLOW (เช็คฟิต)
    2. เช็คว่ามีงบเหลือในธนาคาร (budget_itb) เท่าใด และหากขายตัวเจ็บออก จะมีงบสูงสุดเท่าใด
    3. ใช้เครื่องมือ Query FPL Replacements ค้นหาตัวเลือกทดแทนที่ฟอร์มดีที่สุดและมี ep_next สูงสุด
    """,
    expected_output="สรุปรายชื่อนักเตะที่มีปัญหา และตัวเลือกเข้าใหม่ 3-5 คนพร้อมราคาและสถิติ",
    agent=scout
)

task2 = Task(
    description=f"""
    ข้อมูลทีมปัจจุบัน (JSON):
    {team_json_str}

    จากผลการวิเคราะห์ของ Scout และข้อมูลทีมข้างต้น:
    1. สรุปคำแนะนำการเปลี่ยนตัว (ย้ายออก - ย้ายเข้า หรือ Roll Transfer เก็บโควตา)
    2. แนะนำการใช้ชิป (Wildcard / Free Hit) หากสถานการณ์ทีมวิกฤต
    3. จัดทีมตัวจริง 11 คน (Starting XI) พร้อมจัดรูปแบบแผน (เช่น 3-4-3 หรือ 3-5-2)
    4. จัดลำดับตัวสำรอง 4 คน (GK สำรอง + ม้านั่งลำดับ 1, 2, 3)
    5. ฟันธง Captain (C) และ Vice-Captain (VC) พร้อมเหตุผลสั้นๆ
    """,
    expected_output="แผนปฏิบัติการจัดทีม FPL รายสัปดาห์ในรูปแบบ Markdown อ่านง่าย ชัดเจน",
    agent=manager,
    context=[task1]
)

fpl_crew = Crew(
    agents=[scout, manager],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=True
)


def run_crew_sync(crew):
    """รัน Crew ใน thread แยก เพราะ Jupyter มี asyncio event loop ทำงานอยู่แล้ว
    ถ้าเรียก crew.kickoff() ตรงๆ จะได้ RuntimeError: Agent execution was invoked
    synchronously from within a running event loop"""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result()


print("⚡ เริ่มการรัน CrewAI Execution...")
_started_at = time.time()
final_result = run_crew_sync(fpl_crew)
print(f"✅ Crew ทำงานเสร็จใน {time.time() - _started_at:.1f} วินาที")


⚡ เริ่มการรัน CrewAI Execution...


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.33                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 23b56043-7a52-44d4-98d0-65bea7b47f17                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      วิเคราะห์สภาพทีมจากข้อมูล JSON นี้:                                                                               │
│      {                                                                                                          │
│    "gameweek": "Gameweek 9",                                                                                    │
│    "deadline": "Sat 31 Oct, 18:00",                                                                             │
│    "budget_itb": 1.3,                                                                                           │
│    "free_transfers": 1,                                                                                         │
│    "cost_pts": 0,                                                                                               │
│    "wildcard_available": true,                                                                                  │
│    "free_hit_available": true,                                                                                  │
│    "goalkeepers": [                                                                                             │
│      {                                                                                                          │
│        "name": "Verbruggen",                                                                                    │
│        "position": "GKP",                                                                                       │
│        "price": 4.5,                                                                                            │
│        "fixture": "MCI (A)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "name": "Kinsky",                                                                                        │
│        "position": "GKP",                                                                                       │
│        "price": 4.4,                                                                                            │
│        "fixture": "CRY (H)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "defenders": [                                                                                               │
│      {                                                                                                          │
│        "name": "Calafiori",                                                                                     │
│        "position": "DEF",                                                                                       │
│        "price": 5.7,                                                                                            │
│        "fixture": "LIV (A)",                    

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: FPL Data Analyst & Scout                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│      วิเคราะห์สภาพทีมจากข้อมูล JSON นี้:                                                                               │
│      {                                                                                                          │
│    "gameweek": "Gameweek 9",                                                                                    │
│    "deadline": "Sat 31 Oct, 18:00",                                                                             │
│    "budget_itb": 1.3,                                                                                           │
│    "free_transfers": 1,                                                                                         │
│    "cost_pts": 0,                                                                                               │
│    "wildcard_available": true,                                                                                  │
│    "free_hit_available": true,                                                                                  │
│    "goalkeepers": [                                                                                             │
│      {                                                                                                          │
│        "name": "Verbruggen",                                                                                    │
│        "position": "GKP",                                                                                       │
│        "price": 4.5,                                                                                            │
│        "fixture": "MCI (A)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "name": "Kinsky",                                                                                        │
│        "position": "GKP",                                                                                       │
│        "price": 4.4,                                                                                            │
│        "fixture": "CRY (H)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "defenders": [                                                                                               │
│      {                                                                                                          │
│        "name": "Calafiori",                                                                                     │
│        "position": "DEF",                                                                                       │
│        "price": 5.7,                            

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: query_fpl_replacements                                                                                   │
│  Args: {'position': 'FWD', 'max_price': 8.9}                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (13 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Elite FPL Manager                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 📋 แผนปฏิบัติการ FPL – Gameweek 9                                                                               │
│  **Deadline:** Sat 31 Oct, 18:00                                                                                │
│  **สถานะทีม:** งบเหลือ 1.3m | FT 1 | Cost Points 0 | Wildcard พร้อมใช้ | Free Hit พร้อมใช้                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1) 🔁 คำแนะนำการเปลี่ยนตัว                                                                                     │
│                                                                                                                 │
│  ### ✅ ทำการเปลี่ยนตัว 1 ครั้ง ด้วย Free Transfer                                                                    │
│  | ย้ายออก | ย้ายเข้า | เหตุผล |                                                                                    │
│  |---|---|---|                                                                                                  │
│  | **João Pedro** (FWD 7.6m, RED) | **Evanilson** (FWD 6.4m, BOU) | João Pedro แขวน/เจ็บหนัก ไม่มีโอกาสลงเล่น        │
│  ต้องปลดทุน 7.6m ออกทันที ส่วน Evanilson เป็นตัวเลือกที่ form 8.5 และ EP Next 8.5 สูงสุดในกลุ่มงบไม่เกิน 8.9m |                 │
│                                                                                                                 │
│  **ผลลัพธ์หลังเปลี่ยนตัว:**                                                                                           │
│  - ใช้ FT 1 ครั้ง → **เหลือ Free Transfer 0**                                                                       │
│  - งบใหม่: 1.3 + 7.6 − 6.4 = **เหลือ 2.5m**                                                                       │
│  - Cost Points: **0**                                                                                           │
│  - ไม่ต้องเสียคะแนนติดลบ                                                                                            │
│                                                                                                                 │
│  ### 🟡 Calafiori – เก็บไว้ก่อน แต่ไม่เริ่ม                                                                            │
│  - สถานะ YELLOW = ยังมีโอกาสลงเล่นประมาณ 75%                                                                       │
│  - แต่เจอ **LIV (A)** ซึ่งเป็น fixture แย่ที่สุดในทีม                                                                    │
│  - แผนรัดกุม: **เก็บ Calafiori ไว้เป็นตัวสำรอง** ไม่ขายทิ้ง และไม่เอาเข้าตัวจริง                                             │
│  - ถ้าก่อน deadline Calafiori กลายเป็น RED จริง: **ไม่ต้องรีบเปลี่ยนเพิ่ม** เพราะมี Aina/O’Shea เป็นตัวจริงอยู่แล้ว —             │
│  ไม่แนะนำให้เสีย −4 คะแนน                                                                                          │
│                                                                                                                 │
│  ### ❌ ไม่ Roll Transfer                                                                                        │
│  - แม้จะเหลือ 1 FT แต่การเก็บ João Pedro (RED) ไว้เท่ากับเสียเง

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      ข้อมูลทีมปัจจุบัน (JSON):                                                                                        │
│      {                                                                                                          │
│    "gameweek": "Gameweek 9",                                                                                    │
│    "deadline": "Sat 31 Oct, 18:00",                                                                             │
│    "budget_itb": 1.3,                                                                                           │
│    "free_transfers": 1,                                                                                         │
│    "cost_pts": 0,                                                                                               │
│    "wildcard_available": true,                                                                                  │
│    "free_hit_available": true,                                                                                  │
│    "goalkeepers": [                                                                                             │
│      {                                                                                                          │
│        "name": "Verbruggen",                                                                                    │
│        "position": "GKP",                                                                                       │
│        "price": 4.5,                                                                                            │
│        "fixture": "MCI (A)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "name": "Kinsky",                                                                                        │
│        "position": "GKP",                                                                                       │
│        "price": 4.4,                                                                                            │
│        "fixture": "CRY (H)",                                                                                    │
│        "status_flag": "NONE"                                                                                    │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "defenders": [                                                                                               │
│      {                                                                                                          │
│        "name": "Calafiori",                                                                                     │
│        "position": "DEF",                                                                                       │
│        "price": 5.7,                                                                                            │
│        "fixture": "LIV (A)",                     

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 23b56043-7a52-44d4-98d0-65bea7b47f17                                                                       │
│  Final Output: # 📋 แผนปฏิบัติการ FPL – Gameweek 9                                                                 │
│  **Deadline:** Sat 31 Oct, 18:00                                                                                │
│  **สถานะทีม:** งบเหลือ 1.3m | FT 1 | Cost Points 0 | Wildcard พร้อมใช้ | Free Hit พร้อมใช้                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1) 🔁 คำแนะนำการเปลี่ยนตัว                                                                                     │
│                                                                                                                 │
│  ### ✅ ทำการเปลี่ยนตัว 1 ครั้ง ด้วย Free Transfer                                                                    │
│  | ย้ายออก | ย้ายเข้า | เหตุผล |                                                                                    │
│  |---|---|---|                                                                                                  │
│  | **João Pedro** (FWD 7.6m, RED) | **Evanilson** (FWD 6.4m, BOU) | João Pedro แขวน/เจ็บหนัก ไม่มีโอกาสลงเล่น        │
│  ต้องปลดทุน 7.6m ออกทันที ส่วน Evanilson เป็นตัวเลือกที่ form 8.5 และ EP Next 8.5 สูงสุดในกลุ่มงบไม่เกิน 8.9m |                 │
│                                                                                                                 │
│  **ผลลัพธ์หลังเปลี่ยนตัว:**                                                                                           │
│  - ใช้ FT 1 ครั้ง → **เหลือ Free Transfer 0**                                                                       │
│  - งบใหม่: 1.3 + 7.6 − 6.4 = **เหลือ 2.5m**                                                                       │
│  - Cost Points: **0**                                                                                           │
│  - ไม่ต้องเสียคะแนนติดลบ                                                                                            │
│                                                                                                                 │
│  ### 🟡 Calafiori – เก็บไว้ก่อน แต่ไม่เริ่ม                                                                            │
│  - สถานะ YELLOW = ยังมีโอกาสลงเล่นประมาณ 75%                                                                       │
│  - แต่เจอ **LIV (A)** ซึ่งเป็น fixture แย่ที่สุดในทีม                                                                    │
│  - แผนรัดกุม: **เก็บ Calafiori ไว้เป็นตัวสำรอง** ไม่ขายทิ้ง และไม่เอาเข้าตัวจริง                                             │
│  - ถ้าก่อน deadline Calafiori กลายเป็น RED จริง: **ไม่ต้องรีบเปลี่ยนเพิ่ม** เพราะมี Aina/O’Shea เป็นตัวจริงอยู่แล้ว —             │
│  ไม่แนะนำให้เสีย −4 คะแนน                                                                                          │
│                                                                                                                 │
│  ### ❌ ไม่ Roll Transfer                                                                                        │
│  - แม้จะเหลือ 1 FT แต่การเก็บ João Pedro (RED) ไว้เท่ากับเสียเ

✅ Crew ทำงานเสร็จใน 41.0 วินาที


## 🏆 Step 6: แสดงผลลัพธ์สุดท้าย (Final Strategy & Lineup)
นำเสนอผลลัพธ์ในรูปแบบ Markdown ที่สวยงาม

In [8]:
from IPython.display import display, Markdown

display(Markdown("# 📋 สรุปแผนการจัดทีม FPL ประจำสัปดาห์"))
display(Markdown(str(final_result)))

# 📋 สรุปแผนการจัดทีม FPL ประจำสัปดาห์

# 📋 แผนปฏิบัติการ FPL – Gameweek 9  
**Deadline:** Sat 31 Oct, 18:00  
**สถานะทีม:** งบเหลือ 1.3m | FT 1 | Cost Points 0 | Wildcard พร้อมใช้ | Free Hit พร้อมใช้

---

## 1) 🔁 คำแนะนำการเปลี่ยนตัว

### ✅ ทำการเปลี่ยนตัว 1 ครั้ง ด้วย Free Transfer
| ย้ายออก | ย้ายเข้า | เหตุผล |
|---|---|---|
| **João Pedro** (FWD 7.6m, RED) | **Evanilson** (FWD 6.4m, BOU) | João Pedro แขวน/เจ็บหนัก ไม่มีโอกาสลงเล่น ต้องปลดทุน 7.6m ออกทันที ส่วน Evanilson เป็นตัวเลือกที่ form 8.5 และ EP Next 8.5 สูงสุดในกลุ่มงบไม่เกิน 8.9m |

**ผลลัพธ์หลังเปลี่ยนตัว:**
- ใช้ FT 1 ครั้ง → **เหลือ Free Transfer 0**
- งบใหม่: 1.3 + 7.6 − 6.4 = **เหลือ 2.5m**
- Cost Points: **0**
- ไม่ต้องเสียคะแนนติดลบ

### 🟡 Calafiori – เก็บไว้ก่อน แต่ไม่เริ่ม
- สถานะ YELLOW = ยังมีโอกาสลงเล่นประมาณ 75%
- แต่เจอ **LIV (A)** ซึ่งเป็น fixture แย่ที่สุดในทีม
- แผนรัดกุม: **เก็บ Calafiori ไว้เป็นตัวสำรอง** ไม่ขายทิ้ง และไม่เอาเข้าตัวจริง
- ถ้าก่อน deadline Calafiori กลายเป็น RED จริง: **ไม่ต้องรีบเปลี่ยนเพิ่ม** เพราะมี Aina/O’Shea เป็นตัวจริงอยู่แล้ว — ไม่แนะนำให้เสีย −4 คะแนน

### ❌ ไม่ Roll Transfer
- แม้จะเหลือ 1 FT แต่การเก็บ João Pedro (RED) ไว้เท่ากับเสียเงิน 7.6m ไปเปล่า ๆ
- ใช้ FT แก้จุดวิกฤตที่สุดก่อน คือกองหน้า RED

---

## 2) 🎟️ การใช้ชิป

| ชิป | คำแนะนำ | เหตุผล |
|---|---|---|
| **Wildcard** | **ยังไม่ใช้** | หลังเปลี่ยน João Pedro ทีมยังจัด 11 ตัวที่ลงเล่นได้ และมีตัวสำรองที่พอใช้ได้ การเปิด WC กับปัญหา 1–2 จุดถือว่าสิ้นเปลือง |
| **Free Hit** | **ยังไม่ใช้** | เก็บไว้ใช้กับ GW ที่มี blank/double หรือทีมเจอวิกฤตหลายตำแหน่งพร้อมกัน |

> **เงื่อนไขฉุกเฉิน:** ถ้าก่อน deadline มีนักเตะตัวจริงถูกประกาศเป็น RED 2 คนขึ้นไป หรือ Calafiori กลายเป็น RED และทีมเหลือตัวเลือกไม่พอ ให้พิจารณาเปิด **Wildcard** เพื่อแก้หลายจุดพร้อมกัน แต่ตามข้อมูลปัจจุบัน **ไม่จำเป็น**

---

## 3) ⚽ Starting XI – แผน 4-4-2

| ตำแหน่ง | นักเตะ | นัดถัดไป | หมายเหตุ |
|---|---|---|---|
| GK | **Kinsky** | CRY (H) | เลือกเหนือ Verbruggen เพราะเจอ MCI (A) โหดกว่า |
| DEF | **Hall** | EVE (H) | fixture ดี มีโอกาสทำแต้ม |
| DEF | **Ajer** | NFO (H) | เจอในบ้าน ปลอดภัย |
| DEF | **O’Shea** | HUL (A) | ตัวจริงราคาถูก fixture พอใช้ |
| DEF | **Aina** | BRE (A) | ลงเล่นแน่ ให้โอกาสก่อน Calafiori ที่ติดเหลือง |
| MID | **Schade** | NFO (H) | fixture ดี น่าจะเป็นตัวทำแต้มหลักใน midfield |
| MID | **B.Fernandes** | CHE (A) | แข็ง แต่เป็นตัวหลักและมีโอกาสเลี้ยงแต้ม |
| MID | **Mbeumo** | CHE (A) | เช่นเดียวกับ Fernandes ต้องพึ่งคุณภาพindividual |
| MID | **M.Sangaré** | NFO (H) | fixture ในบ้านดีกว่า I.Sangaré ที่ไป BRE (A) |
| FWD | **Haaland (C)** | BHA (H) | ตัวจบสกอร์สูงสุด เจอในบ้าน |
| FWD | **Evanilson** | BOU | ตัวใหม่ form 8.5 / EP Next 8.5 สูงสุดในทีมรองจาก Haaland |

### 📌 หมายเหตุตัวจริง
- **ไม่เริ่ม Calafiori** เพราะติด YELLOW + เจอ Liverpool นอกบ้าน
- **ไม่เริ่ม Kostoulas** เพราะเจอ MCI (A) โหดมาก
- **ไม่เริ่ม I.Sangaré** เพราะเจอ BRE (A) ขณะที่ M.Sangaré เจอ NFO (H) ดีกว่า

---

## 4) 🪑 ลำดับตัวสำรอง

| ลำดับ | นักเตะ | ตำแหน่ง | เหตุผล |
|---|---|---|---|
| **GK สำรอง** | **Verbruggen** | GKP | เจอ MCI (A) จึงเป็นสำรองของ Kinsky |
| **Bench 1** | **Calafiori** | DEF | ถ้าฟิตทันมี attacking upside แม้เจอ LIV (A) — ถ้าไม่เล่น autosub จะข้ามไปคนถัดไป |
| **Bench 2** | **I.Sangaré** | MID | มีโอกาสลงเล่น เจอ BRE (A) |
| **Bench 3** | **Kostoulas** | FWD | เจอ MCI (A) โอกาสแต้มต่ำ |

**ลำดับ autosub ที่แนะนำ:** Calafiori → I.Sangaré → Kostoulas

---

## 5) 🧢 Captain & Vice-Captain

| บทบาท | นักเตะ | เหตุผล |
|---|---|---|
| **Captain (C)** | **Haaland** | เจอ BHA (H) เป็น fixture ดีที่สุดในทีม และเป็นตัวจบสกอร์ระดับสูงสุด — เพดานคะแนนสูงที่สุด |
| **Vice-Captain (VC)** | **Evanilson** | form 8.5 และ EP Next 8.5 สูงสุดในกลุ่มตัวเลือกหลัง João Pedro ออก เป็นกองหน้าตัวหลักที่พร้อมลง |

> ไม่เลือก Fernandes/Mbeumo เป็น VC เพราะทั้งคู่เจอ Chelsea (A) พร้อมกัน ซึ่งเสี่ยงกว่าการเลือก Evanilson ที่ฟอร์มร้อนกว่า

---

## ✅ สรุปแผนสุดท้าย

- **Transfer:** João Pedro → **Evanilson**
- **ชิป:** ยังไม่ใช้ทั้ง Wildcard และ Free Hit
- **Starting XI (4-4-2):**  
  Kinsky; Hall, Ajer, O’Shea, Aina; Schade, B.Fernandes, Mbeumo, M.Sangaré; Haaland (C), Evanilson (VC)
- **Bench:** Verbruggen; Calafiori, I.Sangaré, Kostoulas
- **Captain:** Haaland
- **Vice-Captain:** Evanilson